# App-14 - Connect Four : Benchmark Adversarial Search

**Navigation** : [<< App-12 ConnectFour](App-12-ConnectFour.ipynb) | [Index](../../README.md) | [CSP Applications >>](../CSP/App-1-NQueens.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comparer** les algorithmes Minimax, Alpha-Beta et MCTS sur un jeu reel
2. **Mesurer** les compromis entre temps de calcul, noeuds explores et qualite de jeu
3. **Analyser** l'impact de la profondeur de recherche et du nombre d'iterations
4. **Choisir** l'algorithme adapte selon les contraintes (temps, precision)

### Prerequis
- [Search-3-Informed](../../Part1-Foundations/Search-3-Informed.ipynb) : Heuristiques
- [Search-6-AdversarialSearch](../../Part1-Foundations/Search-6-AdversarialSearch.ipynb) : Minimax, Alpha-Beta
- [Search-7-MCTS-And-Beyond](../../Part1-Foundations/Search-7-MCTS-And-Beyond.ipynb) : MCTS (recommande)

### Duree estimee : 45 minutes

> **Source** : Adapte des projets etudiants JVX (ECE 2026) et EPITA PPC 2025.
>
> **Note** : Ce notebook se concentre sur le benchmark des trois algorithmes principaux.
> Pour une etude complete avec 8 algorithmes (DQN-RL, etc.), voir [App-12](App-12-ConnectFour.ipynb).

## 1. Introduction

### Pourquoi le Puissance 4 ?

Le **Puissance 4** (Connect Four) est un excellent terrain de test pour les algorithmes de recherche adversariale :

| Caracteristique | Valeur | Impact sur les algorithmes |
|----------------|--------|---------------------------|
| Taille du plateau | 7x6 = 42 cases | Assez grand pour etre interessant |
| Facteur de branchement | ~4-7 | Teste l'efficacite de l'elagage |
| Profondeur moyenne | ~21 coups | Suffisant pour mesurer les performances |
| Jeu resolu | Oui (1988) | Permet de verifier l'optimalite |

### Les trois algorithmes compares

1. **Minimax** : L'algorithme de base, exploration complete de l'arbre
2. **Alpha-Beta** : Minimax avec elagage, beaucoup plus efficace
3. **MCTS** : Approche probabiliste, pas besoin de fonction d'evaluation

### Questions que nous allons repondre

- Quel algorithme est le plus rapide ?
- Lequel joue le mieux avec un temps limite ?
- Comment la profondeur affecte-t-elle la qualite ?

In [ ]:
# Imports
import sys
import time
import random
import math
from typing import Optional, List, Dict, Tuple, Any
from copy import deepcopy
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

print("Environnement pret pour le benchmark adversarial.")

## 2. Implementation du jeu

### Representation du plateau

Nous utilisons une representation efficace :
- Grille 7x6 stockee dans un tableau numpy
- 0 = vide, 1 = joueur MAX (Rouge), -1 = joueur MIN (Jaune)
- Les coups sont indices de colonne (0 a 6)

In [ ]:
class ConnectFour:
    """
    Moteur de jeu Connect Four (Puissance 4).
    
    Representation:
    - Grille 7 colonnes x 6 lignes
    - 0: case vide, 1: joueur MAX (Rouge), -1: joueur MIN (Jaune)
    - Les jetons tombent par gravite
    """
    
    ROWS = 6
    COLS = 7
    WIN_LENGTH = 4
    
    def __init__(self):
        self.board = np.zeros((self.ROWS, self.COLS), dtype=int)
        self.current_player = 1  # 1 = MAX, -1 = MIN
    
    def copy(self) -> 'ConnectFour':
        """Cree une copie du jeu."""
        new_game = ConnectFour()
        new_game.board = self.board.copy()
        new_game.current_player = self.current_player
        return new_game
    
    def get_state(self) -> Tuple[np.ndarray, int]:
        """Retourne l'etat (plateau, joueur)."""
        return self.board.copy(), self.current_player
    
    def set_state(self, state: Tuple[np.ndarray, int]):
        """Definit l'etat du jeu."""
        self.board, self.current_player = state[0].copy(), state[1]
    
    def get_valid_moves(self) -> List[int]:
        """Retourne les colonnes valides (non pleines)."""
        return [c for c in range(self.COLS) if self.board[0, c] == 0]
    
    def drop_piece(self, col: int) -> bool:
        """
        Depose un jeton dans la colonne donnee.
        Retourne True si le coup est valide.
        """
        if col < 0 or col >= self.COLS or self.board[0, col] != 0:
            return False
        
        # Trouver la premiere case vide en partant du bas
        for row in range(self.ROWS - 1, -1, -1):
            if self.board[row, col] == 0:
                self.board[row, col] = self.current_player
                self.current_player *= -1
                return True
        return False
    
    def undo_move(self, col: int):
        """Annule le dernier coup dans la colonne donnee."""
        for row in range(self.ROWS):
            if self.board[row, col] != 0:
                self.board[row, col] = 0
                self.current_player *= -1
                return
    
    def check_winner(self) -> int:
        """
        Verifie s'il y a un gagnant.
        Retourne: 1 (MAX gagne), -1 (MIN gagne), 0 (pas de gagnant)
        """
        # Verifier horizontales
        for r in range(self.ROWS):
            for c in range(self.COLS - 3):
                window = self.board[r, c:c+4]
                if abs(window.sum()) == 4 and len(set(window)) == 1:
                    return int(window[0])
        
        # Verifier verticales
        for r in range(self.ROWS - 3):
            for c in range(self.COLS):
                window = self.board[r:r+4, c]
                if abs(window.sum()) == 4 and len(set(window)) == 1:
                    return int(window[0])
        
        # Verifier diagonales (bas-droite)
        for r in range(self.ROWS - 3):
            for c in range(self.COLS - 3):
                window = [self.board[r+i, c+i] for i in range(4)]
                if abs(sum(window)) == 4 and len(set(window)) == 1:
                    return int(window[0])
        
        # Verifier diagonales (bas-gauche)
        for r in range(3, self.ROWS):
            for c in range(self.COLS - 3):
                window = [self.board[r-i, c+i] for i in range(4)]
                if abs(sum(window)) == 4 and len(set(window)) == 1:
                    return int(window[0])
        
        return 0
    
    def is_terminal(self) -> bool:
        """Le jeu est-il termine ?"""
        return self.check_winner() != 0 or len(self.get_valid_moves()) == 0
    
    def get_utility(self, maximizing_player: int = 1) -> float:
        """
        Retourne l'utilite de l'etat terminal.
        +1 si MAX gagne, -1 si MIN gagne, 0 sinon.
        """
        winner = self.check_winner()
        return float(winner * maximizing_player)
    
    def display(self) -> str:
        """Representation textuelle du plateau."""
        symbols = {0: '.', 1: 'X', -1: 'O'}
        lines = []
        for row in self.board:
            lines.append(' '.join(symbols[cell] for cell in row))
        lines.append(' '.join(str(c) for c in range(self.COLS)))
        return '\n'.join(lines)

# Test du moteur de jeu
game = ConnectFour()
print("Plateau initial:")
print(game.display())
print(f"\nJoueur actuel: {'MAX (X)' if game.current_player == 1 else 'MIN (O)'}")
print(f"Coups valides: {game.get_valid_moves()}")

In [ ]:
# Test d'une partie courte
game = ConnectFour()
moves = [3, 3, 4, 2, 2, 4, 1, 5, 5]  # Sequence de test

for move in moves:
    game.drop_piece(move)

print("Plateau apres quelques coups:")
print(game.display())
print(f"\nGagnant: {game.check_winner()}")

## 3. Algorithme Minimax

### Rappel du principe

Minimax explore l'arbre de jeu complet jusqu'a une profondeur donnee :
- **MAX** cherche a maximiser l'utilite
- **MIN** cherche a minimiser l'utilite

### Fonction d'evaluation

Pour les etats non terminaux, nous utilisons une heuristique basee sur :
- Le controle du centre
- Le nombre de "fenetres" gagnantes potentielles

In [ ]:
def evaluate_window(window: List[int], player: int) -> float:
    """
    Evalue une fenetre de 4 cases.
    Attribue des scores selon le potentiel de victoire.
    """
    score = 0
    opponent = -player
    
    player_count = window.count(player)
    opponent_count = window.count(opponent)
    empty_count = window.count(0)
    
    # Fenetre gagnante
    if player_count == 4:
        score += 100
    # 3 jetons du joueur + 1 vide
    elif player_count == 3 and empty_count == 1:
        score += 5
    # 2 jetons du joueur + 2 vides
    elif player_count == 2 and empty_count == 2:
        score += 2
    
    # Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 and empty_count == 1:
        score -= 4
    
    return score

def evaluate_position(game: ConnectFour, player: int = 1) -> float:
    """
    Fonction d'evaluation heuristique pour Connect Four.
    
    Criteres:
    - Score des fenetres (lignes, colonnes, diagonales)
    - Bonus pour le controle du centre
    """
    if game.is_terminal():
        return game.get_utility(player) * 1000  # Victoire/defaite certaine
    
    score = 0
    board = game.board
    
    # Bonus pour le centre (colonne 3)
    center_array = list(board[:, 3])
    score += center_array.count(player) * 3
    
    # Evaluer les horizontales
    for r in range(game.ROWS):
        for c in range(game.COLS - 3):
            window = list(board[r, c:c+4])
            score += evaluate_window(window, player)
    
    # Evaluer les verticales
    for r in range(game.ROWS - 3):
        for c in range(game.COLS):
            window = list(board[r:r+4, c])
            score += evaluate_window(window, player)
    
    # Evaluer les diagonales (bas-droite)
    for r in range(game.ROWS - 3):
        for c in range(game.COLS - 3):
            window = [board[r+i, c+i] for i in range(4)]
            score += evaluate_window(window, player)
    
    # Evaluer les diagonales (bas-gauche)
    for r in range(3, game.ROWS):
        for c in range(game.COLS - 3):
            window = [board[r-i, c+i] for i in range(4)]
            score += evaluate_window(window, player)
    
    return score

# Test de l'evaluation
game = ConnectFour()
for move in [3, 3, 4]:  # MAX joue au centre
    game.drop_piece(move)

print(game.display())
print(f"\nScore pour MAX: {evaluate_position(game, 1)}")
print(f"Score pour MIN: {evaluate_position(game, -1)}")

In [ ]:
# Statistiques globales pour le benchmark
class SearchStats:
    """Classe pour collecter les statistiques de recherche."""
    def __init__(self):
        self.nodes_explored = 0
        self.start_time = 0
        self.elapsed = 0
    
    def reset(self):
        self.nodes_explored = 0
        self.start_time = time.time()
    
    def stop(self):
        self.elapsed = time.time() - self.start_time

def minimax(game: ConnectFour, depth: int, maximizing: bool, 
            stats: SearchStats, player: int = 1) -> Tuple[float, Optional[int]]:
    """
    Algorithme Minimax avec profondeur limitee.
    
    Args:
        game: Instance du jeu
        depth: Profondeur de recherche restante
        maximizing: True si c'est le tour de MAX
        stats: Objet pour collecter les statistiques
        player: Joueur pour lequel on maximise (1 ou -1)
    
    Returns:
        (valeur, meilleure_action)
    """
    stats.nodes_explored += 1
    
    # Cas terminal
    if depth == 0 or game.is_terminal():
        if game.is_terminal():
            return game.get_utility(player) * 1000, None
        return evaluate_position(game, player), None
    
    valid_moves = game.get_valid_moves()
    
    if maximizing:
        best_value = float('-inf')
        best_move = valid_moves[0] if valid_moves else None
        
        for move in valid_moves:
            game.drop_piece(move)
            value, _ = minimax(game, depth - 1, False, stats, player)
            game.undo_move(move)
            
            if value > best_value:
                best_value = value
                best_move = move
        
        return best_value, best_move
    else:
        best_value = float('+inf')
        best_move = valid_moves[0] if valid_moves else None
        
        for move in valid_moves:
            game.drop_piece(move)
            value, _ = minimax(game, depth - 1, True, stats, player)
            game.undo_move(move)
            
            if value < best_value:
                best_value = value
                best_move = move
        
        return best_value, best_move

# Test Minimax
game = ConnectFour()
stats = SearchStats()
stats.reset()

value, move = minimax(game, depth=4, maximizing=True, stats=stats)
stats.stop()

print(f"Minimax (profondeur 4):")
print(f"  Meilleur coup: {move}")
print(f"  Valeur: {value:.2f}")
print(f"  Noeuds explores: {stats.nodes_explored:,}")
print(f"  Temps: {stats.elapsed:.4f}s")

## 4. Algorithme Alpha-Beta

### Principe de l'elagage

Alpha-Beta ameliore Minimax en eliminant les branches inutiles :
- **alpha** : Meilleure valeur que MAX peut garantir
- **beta** : Meilleure valeur que MIN peut garantir

Si alpha >= beta, on peut couper la branche (l'adversaire ne permettra jamais d'atteindre cette branche).

### Gain theorique

Avec un ordonnancement optimal des coups : **O(b^(d/2))** au lieu de **O(b^d)**.

In [ ]:
def alpha_beta(game: ConnectFour, depth: int, alpha: float, beta: float,
               maximizing: bool, stats: SearchStats, player: int = 1) -> Tuple[float, Optional[int]]:
    """
    Algorithme Alpha-Beta pruning.
    
    Args:
        game: Instance du jeu
        depth: Profondeur de recherche restante
        alpha: Meilleure valeur pour MAX
        beta: Meilleure valeur pour MIN
        maximizing: True si c'est le tour de MAX
        stats: Objet pour collecter les statistiques
        player: Joueur pour lequel on maximise
    
    Returns:
        (valeur, meilleure_action)
    """
    stats.nodes_explored += 1
    
    # Cas terminal
    if depth == 0 or game.is_terminal():
        if game.is_terminal():
            return game.get_utility(player) * 1000, None
        return evaluate_position(game, player), None
    
    valid_moves = game.get_valid_moves()
    
    # Ordonnancement des coups : privilegier le centre
    center = 3
    valid_moves.sort(key=lambda x: abs(x - center))
    
    if maximizing:
        best_value = float('-inf')
        best_move = valid_moves[0] if valid_moves else None
        
        for move in valid_moves:
            game.drop_piece(move)
            value, _ = alpha_beta(game, depth - 1, alpha, beta, False, stats, player)
            game.undo_move(move)
            
            if value > best_value:
                best_value = value
                best_move = move
            
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break  # Elagage beta
        
        return best_value, best_move
    else:
        best_value = float('+inf')
        best_move = valid_moves[0] if valid_moves else None
        
        for move in valid_moves:
            game.drop_piece(move)
            value, _ = alpha_beta(game, depth - 1, alpha, beta, True, stats, player)
            game.undo_move(move)
            
            if value < best_value:
                best_value = value
                best_move = move
            
            beta = min(beta, best_value)
            if beta <= alpha:
                break  # Elagage alpha
        
        return best_value, best_move

# Test Alpha-Beta
game = ConnectFour()
stats_ab = SearchStats()
stats_ab.reset()

value_ab, move_ab = alpha_beta(game, depth=4, alpha=float('-inf'), beta=float('+inf'),
                               maximizing=True, stats=stats_ab)
stats_ab.stop()

print(f"Alpha-Beta (profondeur 4):")
print(f"  Meilleur coup: {move_ab}")
print(f"  Valeur: {value_ab:.2f}")
print(f"  Noeuds explores: {stats_ab.nodes_explored:,}")
print(f"  Temps: {stats_ab.elapsed:.4f}s")
print(f"\nReduction vs Minimax: {stats.nodes_explored / stats_ab.nodes_explored:.1f}x")

## 5. Algorithme MCTS

### Principe de Monte Carlo Tree Search

MCTS construit progressivement un arbre de recherche en equilibant exploration et exploitation :

1. **Selection** : Descendre dans l'arbre avec UCB1
2. **Expansion** : Ajouter un nouveau noeud
3. **Simulation** : Jouer une partie aleatoire (rollout)
4. **Backpropagation** : Remonter le resultat

### Formule UCB1

$$UCB1 = \frac{W}{N} + c \sqrt{\frac{\ln(N_{parent})}{N}}$$

Ou :
- W = nombre de victoires
- N = nombre de visites
- c = parametre d'exploration (typiquement 1.41)

In [ ]:
class MCTSNode:
    """Noeud de l'arbre MCTS."""
    
    def __init__(self, game_state: Tuple[np.ndarray, int], parent=None, action=None):
        self.state = game_state
        self.parent = parent
        self.action = action
        self.children: Dict[int, 'MCTSNode'] = {}
        self.visits = 0
        self.wins = 0.0
        self.untried_actions = None
    
    def ucb1(self, c: float = 1.41) -> float:
        """Calcule le score UCB1."""
        if self.visits == 0:
            return float('inf')
        
        exploitation = self.wins / self.visits
        exploration = c * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration
    
    def best_child_ucb1(self, c: float = 1.41) -> 'MCTSNode':
        """Selectionne l'enfant avec le meilleur UCB1."""
        return max(self.children.values(), key=lambda n: n.ucb1(c))
    
    def best_child_visits(self) -> 'MCTSNode':
        """Selectionne l'enfant le plus visite (pour le coup final)."""
        return max(self.children.values(), key=lambda n: n.visits)

class MCTS:
    """Implementation de Monte Carlo Tree Search pour Connect Four."""
    
    def __init__(self, c: float = 1.41):
        self.c = c
        self.stats = SearchStats()
    
    def search(self, game: ConnectFour, iterations: int = 1000) -> Tuple[Optional[int], float]:
        """
        Execute la recherche MCTS.
        
        Returns:
            (meilleure_action, taux_de_victoire_estime)
        """
        self.stats.reset()
        root = MCTSNode(game.get_state())
        
        for _ in range(iterations):
            node = self._select(root, game)
            result = self._simulate(game)
            self._backpropagate(node, result)
        
        self.stats.stop()
        
        if not root.children:
            return None, 0.0
        
        best = root.best_child_visits()
        win_rate = best.wins / best.visits if best.visits > 0 else 0
        return best.action, win_rate
    
    def _select(self, node: MCTSNode, game: ConnectFour) -> MCTSNode:
        """Selection : descend dans l'arbre avec UCB1."""
        while not self._is_terminal(node.state, game):
            if not self._is_fully_expanded(node, game):
                return self._expand(node, game)
            else:
                node = node.best_child_ucb1(self.c)
                game.set_state(node.state)
        
        return node
    
    def _expand(self, node: MCTSNode, game: ConnectFour) -> MCTSNode:
        """Expansion : ajoute un nouveau noeud."""
        self.stats.nodes_explored += 1
        
        if node.untried_actions is None:
            game.set_state(node.state)
            node.untried_actions = game.get_valid_moves().copy()
        
        if not node.untried_actions:
            return node
        
        action = node.untried_actions.pop()
        game.set_state(node.state)
        game.drop_piece(action)
        
        child = MCTSNode(game.get_state(), parent=node, action=action)
        node.children[action] = child
        return child
    
    def _simulate(self, game: ConnectFour) -> float:
        """Simulation : joue une partie aleatoire."""
        self.stats.nodes_explored += 1
        
        # Sauvegarder l'etat
        saved_state = game.get_state()
        player = saved_state[1]
        original_player = player
        
        while not game.is_terminal():
            moves = game.get_valid_moves()
            if not moves:
                break
            move = random.choice(moves)
            game.drop_piece(move)
        
        result = game.get_utility(original_player)
        
        # Restaurer l'etat
        game.set_state(saved_state)
        return result
    
    def _backpropagate(self, node: MCTSNode, result: float):
        """Backpropagation : remonte le resultat."""
        while node is not None:
            node.visits += 1
            # Adapter le resultat au point de vue du joueur a ce noeud
            # result est du point de vue du joueur original (MAX)
            node.wins += result if node.state[1] == -1 else -result  # Inverser si c'est MIN
            node = node.parent
    
    def _is_terminal(self, state: Tuple[np.ndarray, int], game: ConnectFour) -> bool:
        """Verifie si l'etat est terminal."""
        game.set_state(state)
        return game.is_terminal()
    
    def _is_fully_expanded(self, node: MCTSNode, game: ConnectFour) -> bool:
        """Verifie si toutes les actions ont ete expandues."""
        if node.untried_actions is None:
            game.set_state(node.state)
            node.untried_actions = game.get_valid_moves().copy()
        return len(node.untried_actions) == 0 and len(node.children) > 0

# Test MCTS
game = ConnectFour()
mcts = MCTS(c=1.41)

move_mcts, win_rate = mcts.search(game, iterations=1000)

print(f"MCTS (1000 iterations):")
print(f"  Meilleur coup: {move_mcts}")
print(f"  Taux de victoire estime: {win_rate:.3f}")
print(f"  Noeuds explores: {mcts.stats.nodes_explored:,}")
print(f"  Temps: {mcts.stats.elapsed:.4f}s")

## 6. Benchmark Comparatif

Comparons les trois algorithmes sur differentes dimensions :
- **Temps d'execution**
- **Noeuds explores**
- **Qualite du coup** (vs coup optimal)

In [ ]:
def run_benchmark_depth(max_depth: int = 6):
    """
    Compare Minimax et Alpha-Beta a differentes profondeurs.
    """
    results = []
    game = ConnectFour()
    
    for depth in range(1, max_depth + 1):
        # Minimax
        stats_mm = SearchStats()
        stats_mm.reset()
        v_mm, m_mm = minimax(game, depth, True, stats_mm)
        stats_mm.stop()
        
        # Alpha-Beta
        stats_ab = SearchStats()
        stats_ab.reset()
        v_ab, m_ab = alpha_beta(game, depth, float('-inf'), float('+inf'), True, stats_ab)
        stats_ab.stop()
        
        results.append({
            'Profondeur': depth,
            'Minimax noeuds': stats_mm.nodes_explored,
            'Minimax temps': stats_mm.elapsed,
            'Alpha-Beta noeuds': stats_ab.nodes_explored,
            'Alpha-Beta temps': stats_ab.elapsed,
            'Speedup noeuds': stats_mm.nodes_explored / max(1, stats_ab.nodes_explored),
            'Speedup temps': stats_mm.elapsed / max(0.0001, stats_ab.elapsed)
        })
    
    return pd.DataFrame(results)

# Executer le benchmark
df_depth = run_benchmark_depth(max_depth=6)
print("Benchmark par profondeur:")
display(df_depth)

In [ ]:
def run_benchmark_mcts(iterations_list: List[int] = [100, 500, 1000, 2000, 5000]):
    """
    Teste MCTS avec differents nombres d'iterations.
    """
    results = []
    game = ConnectFour()
    
    for iterations in iterations_list:
        mcts = MCTS(c=1.41)
        move, win_rate = mcts.search(game, iterations=iterations)
        
        results.append({
            'Iterations': iterations,
            'Coup': move,
            'Taux victoire': win_rate,
            'Noeuds explores': mcts.stats.nodes_explored,
            'Temps (s)': mcts.stats.elapsed
        })
    
    return pd.DataFrame(results)

# Executer le benchmark MCTS
df_mcts = run_benchmark_mcts()
print("Benchmark MCTS:")
display(df_mcts)

In [ ]:
# Visualisation des resultats
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1: Noeuds explores par profondeur
ax1 = axes[0]
ax1.plot(df_depth['Profondeur'], df_depth['Minimax noeuds'], 'o-', label='Minimax', color='#ff6b6b', linewidth=2)
ax1.plot(df_depth['Profondeur'], df_depth['Alpha-Beta noeuds'], 's-', label='Alpha-Beta', color='#4ecdc4', linewidth=2)
ax1.set_xlabel('Profondeur')
ax1.set_ylabel('Noeuds explores')
ax1.set_title('Comparaison Minimax vs Alpha-Beta')
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Graphique 2: Temps d'execution
ax2 = axes[1]
ax2.plot(df_depth['Profondeur'], df_depth['Minimax temps'], 'o-', label='Minimax', color='#ff6b6b', linewidth=2)
ax2.plot(df_depth['Profondeur'], df_depth['Alpha-Beta temps'], 's-', label='Alpha-Beta', color='#4ecdc4', linewidth=2)
ax2.set_xlabel('Profondeur')
ax2.set_ylabel('Temps (secondes)')
ax2.set_title('Temps d\'execution')
ax2.legend()
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_adversarial.png', dpi=150, bbox_inches='tight')
plt.show()

print("Graphique sauvegarde: benchmark_adversarial.png")

### Interpretation des resultats

| Algorithme | Points forts | Points faibles | Usage recommande |
|------------|--------------|----------------|------------------|
| **Minimax** | Simple, optimal | Explosion combinatoire | Petits jeux, enseignement |
| **Alpha-Beta** | Efficace, optimal | Ordonnancement critique | Jeux a branchement moyen |
| **MCTS** | Pas d'heuristique, parallele | Probabiliste | Grands espaces, jeux complexes |

**Observations cles** :

1. **Alpha-Beta** reduit drastiquement le nombre de noeuds (facteur 5-20x selon la profondeur)
2. **L'ordonnancement** des coups est crucial pour maximiser l'elagage
3. **MCTS** converge vers de bons coups sans fonction d'evaluation explicite
4. Pour un temps donne, Alpha-Beta profond bat souvent MCTS sur Connect Four

## 7. Tournoi entre algorithmes

Faisons jouer les algorithmes les uns contre les autres pour mesurer leur force relative.

In [ ]:
def play_game(agent1, agent2, verbose: bool = False) -> int:
    """
    Fait jouer deux agents l'un contre l'autre.
    
    Args:
        agent1: Fonction (game) -> move pour le joueur MAX
        agent2: Fonction (game) -> move pour le joueur MIN
        verbose: Afficher la partie
    
    Returns:
        1 si agent1 gagne, -1 si agent2 gagne, 0 si nul
    """
    game = ConnectFour()
    
    while not game.is_terminal():
        if game.current_player == 1:
            move = agent1(game)
        else:
            move = agent2(game)
        
        if move is None or move not in game.get_valid_moves():
            # Coup invalide = defaite
            return -game.current_player
        
        game.drop_piece(move)
        
        if verbose:
            print(f"Joueur {'X' if game.current_player == -1 else 'O'} joue {move}")
            print(game.display())
            print()
    
    return game.check_winner()

# Definition des agents
def agent_random(game):
    moves = game.get_valid_moves()
    return random.choice(moves) if moves else None

def agent_minimax_depth4(game):
    stats = SearchStats()
    _, move = minimax(game, depth=4, maximizing=(game.current_player == 1), stats=stats)
    return move

def agent_alphabeta_depth4(game):
    stats = SearchStats()
    _, move = alpha_beta(game, depth=4, alpha=float('-inf'), beta=float('+inf'),
                         maximizing=(game.current_player == 1), stats=stats)
    return move

def agent_mcts_1000(game):
    mcts = MCTS(c=1.41)
    move, _ = mcts.search(game, iterations=1000)
    return move

print("Agents definis: Random, Minimax(d=4), AlphaBeta(d=4), MCTS(1000)")

In [ ]:
def run_tournament(agents: Dict[str, callable], n_games: int = 10) -> pd.DataFrame:
    """
    Organise un tournoi round-robin entre les agents.
    """
    results = defaultdict(lambda: {'wins': 0, 'losses': 0, 'draws': 0})
    
    agent_names = list(agents.keys())
    
    for i, name1 in enumerate(agent_names):
        for name2 in agent_names[i+1:]:
            print(f"Match: {name1} vs {name2}...", end=" ")
            
            wins1, wins2, draws = 0, 0, 0
            
            for game_idx in range(n_games):
                # Alterner qui commence
                if game_idx < n_games // 2:
                    result = play_game(agents[name1], agents[name2])
                    if result == 1:
                        wins1 += 1
                    elif result == -1:
                        wins2 += 1
                    else:
                        draws += 1
                else:
                    result = play_game(agents[name2], agents[name1])
                    if result == 1:
                        wins2 += 1
                    elif result == -1:
                        wins1 += 1
                    else:
                        draws += 1
            
            results[name1]['wins'] += wins1
            results[name1]['losses'] += wins2
            results[name1]['draws'] += draws
            results[name2]['wins'] += wins2
            results[name2]['losses'] += wins1
            results[name2]['draws'] += draws
            
            print(f"{wins1}-{wins2}-{draws}")
    
    # Creer le DataFrame
    df = pd.DataFrame([
        {'Agent': name, 'Victoires': r['wins'], 'Defaites': r['losses'], 
         'Nuls': r['draws'], 'Win Rate': r['wins'] / (r['wins'] + r['losses'] + r['draws']) * 100}
        for name, r in results.items()
    ])
    
    return df.sort_values('Win Rate', ascending=False)

# Tournoi reduit (moins de parties pour la demonstration)
agents = {
    'Random': agent_random,
    'Minimax(d=4)': agent_minimax_depth4,
    'AlphaBeta(d=4)': agent_alphabeta_depth4,
    'MCTS(1000)': agent_mcts_1000
}

print("Lancement du tournoi (6 parties par match)...")
df_tournament = run_tournament(agents, n_games=6)
print("\nResultats du tournoi:")
display(df_tournament)

### Analyse du tournoi

Les resultats du tournoi montrent la hierarchie des algorithmes :

1. **Alpha-Beta et Minimax** devraient dominer Random facilement
2. **Alpha-Beta vs Minimax** : Resultats similaires (meme qualite de jeu)
3. **MCTS** : Performance variable selon le nombre d'iterations

> **Note** : Avec seulement 6 parties par match, les resultats peuvent varier.
> Pour une evaluation robuste, utiliser 50+ parties.

## 8. Conclusion

### Synthese comparative

| Critere | Minimax | Alpha-Beta | MCTS |
|---------|---------|------------|------|
| **Optimalite** | Oui (si profondeur complete) | Oui (identique a Minimax) | Probabiliste |
| **Complexite** | O(b^d) | O(b^(d/2)) optimal | O(n * d) |
| **Fonction d'evaluation** | Necessaire | Necessaire | Non requise |
| **Parallelisation** | Difficile | Difficile | Tres facile |
| **Meilleur usage** | Petits jeux | Jeux moyens | Grands jeux |

### Recommandations pratiques

1. **Pour Connect Four** : Alpha-Beta avec profondeur 6-8 et bonne heuristique
2. **Pour les echecs** : Alpha-Beta avec tables de transposition
3. **Pour le Go** : MCTS + reseaux de neurones (AlphaGo)
4. **Pour un temps limite strict** : MCTS ou iterative deepening

### Pour aller plus loin

- **Search-6** : Approfondir Alpha-Beta, transposition tables
- **Search-7** : MCTS avance, OpenSpiel, AlphaZero
- **App-12** : 8 algorithmes compares sur Connect Four
- **GameTheory** : Jeux a information imparfaite, Nash equilibrium

---

**Navigation** : [<< App-12 ConnectFour](App-12-ConnectFour.ipynb) | [Index](../../README.md) | [CSP Applications >>](../CSP/App-1-NQueens.ipynb)

## Exercices

1. **Profondeur variable** : Comparez Alpha-Beta aux profondeurs 4, 6, 8. A quel point la qualite s'ameliorre-t-elle ?

2. **Parametre MCTS** : Testez differentes valeurs de c (0.5, 1.0, 1.41, 2.0). Quel impact sur l'exploration ?

3. **Heuristique amelioree** : Ajoutez la detection des menaces imminentes (3 alignes) a l'evaluation.

4. **Iterative Deepening** : Implementez une recherche avec limite de temps utilisant iterative deepening.

5. **Tournoi etendu** : Lancez un tournoi avec 50 parties par match pour des resultats plus robustes.

6. **Opening Book** : Creez un petit livre d'ouvertures pour les premiers coups et mesurez l'impact.